工具函数

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
  '''2个数的乘法'''
  return a * b


In [8]:
# 天气查询

import requests
def get_weather(city: str) -> str:
  '''
  调用实时天气API，返回温度及天气状况
  参数：
  city：城市名称 如：北京
  '''
  
  url = "https://eolink.o.apispace.com/456456/weather/v001/now"
  payload = { "areacode" : "101010100" }
  headers = {
      "X-APISpace-Token":"pticly09ghw5ub7vjme6q96xu4abu2tm"
  }
  response=requests.get(url, params=payload, headers=headers)
  return response.text

get_weather("北京")


'{"status":0,"result":{"location":{"areacode":"101010100","name":"北京","country":"中国","path":"北京,北京市,北京市,中国"},"realtime":{"text":"晴","code":"00","temp":30.4,"feels_like":32,"rh":50,"wind_class":"2级","wind_speed":1.9,"wind_dir":"东南风","wind_angle":140,"prec":0.0,"prec_time":"2026-05-14 15:00:00","clouds":7,"vis":8000,"pressure":1003,"dew":18,"uv":4,"weight":4,"brief":"很闷热","detail":"无风狂出汗，心静也不凉"},"last_update":"2026-05-14 16:01"}}'

In [13]:
# 获取城市编码
# pip install pandas
import pandas as pd

city_df = pd.read_csv("city.csv")

def get_city_code(city_name: str) -> int:
  '''
  获取城市编码
  '''
  # 优化匹配区县
  match = city_df[city_df["district"]==city_name]
  if not match.empty:
    return match.iloc[0]["areacode/城市ID"]
  # 匹配城市
  match = city_df[city_df["city"]==city_name]
  if not match.empty:
    return match.iloc[0]["areacode/城市ID"]
  # 匹配省份
  match = city_df[city_df['province'].str.contains(city_name, na=False)]
  if not match.empty:
    return match.iloc[0]["areacode/城市ID"]
  # 默认北京
  return 101010100

city_df.head()

,areacode/城市ID,province_geocode,province,city_geocode,city,district_geocode,district,eng,pinyin,lon,lat,exclude（是否为地级市）
0,101010900,110000,北京市,110100,北京市,110106,丰台,fengtai,fengtai,116.286968,39.863642,0
1,101011000,110000,北京市,110100,北京市,110107,石景山,shijingshan,shijingshan,116.195445,39.914601,0
2,101011400,110000,北京市,110100,北京市,110109,门头沟,mentougou,mentougou,116.105381,39.937183,0
3,101011200,110000,北京市,110100,北京市,110111,房山,fangshan,fangshan,116.139157,39.735535,0
4,101010600,110000,北京市,110100,北京市,110112,通州,tongzhou,tongzhou,116.658603,39.902486,0


In [29]:
# 天气查询
from langchain.tools import tool
import requests

@tool
def get_weather(city: str) -> str:
  '''
  调用实时天气API，返回温度及天气状况
  参数：
  city：城市名称 如：北京
  '''
  
  url = "https://eolink.o.apispace.com/456456/weather/v001/now"
  city_code = get_city_code(city)
  payload = { "areacode" : city_code }
  headers = {
      "X-APISpace-Token":"pticly09ghw5ub7vjme6q96xu4abu2tm"
  }
  response=requests.get(url, params=payload, headers=headers)
  # 将结果转换为json数据
  data = response.json()
  temp = data.get('result').get('realtime').get('temp')
  weather = data.get('result').get('realtime').get('text')
  return f"天气: {weather}, 温度: {temp}℃"

调用工具

In [ ]:
# 智普AI
key = '8408ad1573274e03becf025b7ea8ed70.X1EBZbPLs756lTXm'
import os
from langchain_community.chat_models import ChatZhipuAI

os.environ["ZHIPUAI_API_KEY"] = key

chat = ChatZhipuAI(
  model="glm-4",
  temperature=0.5,
)

In [43]:
from langchain_classic.agents import create_react_agent, AgentExecutor

from langsmith import Client
# 获取提示词
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

# 创建工具对象
tools = [get_weather]

# 创建智能体
agent = create_react_agent(llm=chat, tools=tools, prompt=prompt)

# 创建AgentExecutor，运行智能体
agent_executer = AgentExecutor(agent=agent, tools = tools, verbose=True) # verbose代表输出日志
# 调用智能休
response = agent_executer.invoke({"input": "今天北京天气如何？"})

print(response)




> Entering new AgentExecutor chain...
用户想了解今天北京的天气情况。我需要使用 get_weather 函数来获取北京的实时天气信息。
Action: get_weather
Action Input: 北京
Observ天气: 晴, 温度: 30.2℃我已成功获取到北京的天气信息。根据API返回的结果，今天北京的天气是晴天，温度为30.2℃。
Final Answer: 今天北京天气晴朗，温度为30.2℃。

> Finished chain.
{'input': '今天北京天气如何？', 'output': '今天北京天气晴朗，温度为30.2℃。'}


In [51]:
from langchain.tools import tool
import re

@tool
def multiply(input_str: str) -> int:
  '''2个数的乘积
    args:
      input_str: 输入字符串，格式为a*b
    returns: int  
  '''
  match = re.findall(r'\d+', input_str)
  if len(match) == 2:
    a, b = map(int, match)
  return a * b

In [ ]:
from langchain_classic.agents import create_react_agent, AgentExecutor

from langsmith import Client
# 获取提示词
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

# 创建工具对象
tools = [get_weather, multiply]

# 创建智能体
agent = create_react_agent(llm=chat, tools=tools, prompt=prompt)

# 创建AgentExecutor，运行智能体
agent_executer = AgentExecutor(agent=agent, tools = tools, verbose=True) # verbose代表输出日志
# 调用智能休
response = agent_executer.invoke({"input": "1x2等于几？"})

print(response)



> Entering new AgentExecutor chain...
用户问的是“1x2等于几？”，这是一个数学乘法问题。我应该使用multiply工具来计算这个结果。
Action: multiply
Action Input: 1*2
Observ2I now know the final answer
Final Answer: 2

> Finished chain.
{'input': '1x2等于几？', 'output': '2'}


In [ ]:
test_inputs = [
  '今天北京天气如何？',
  '2x3等于几？',
  '帮我修手机'
]

for question in test_inputs:
  resp = agent_executer.invoke({"input": question})
  print(resp)
